# 🧠 SQL ANALYTICS & INSIGHTS NOTEBOOK  
### Project: **Water_Access_Analytics**  
**Author:** Olise Ebinum  

---

## 🌍 Project Overview  

The **Water Access Analytics** project demonstrates a complete SQL analytical workflow for understanding and improving community water access.  
It captures **data reasoning**, **query logic**, and **actionable insights** from exploration to validation — transforming raw relational data into evidence-based water management strategies.

This analysis served as the backbone for **interactive Power BI dashboards** that visualized water access disparities, contamination risks, and improvement progress across provinces.

---

## 🎯 Objectives  

1. Identify inconsistencies between **auditor** and **surveyor** reports.  
2. Quantify **population access** to safe water per town and province.  
3. Detect **human reporting errors** and field efficiency issues.  
4. Generate **data-driven improvement recommendations**.  
5. Support **decision-making** through SQL-driven insights integrated into Power BI.

---

## 🧱 Database Context  

**Database:** `md_water_services`  

The schema integrates **survey**, **audit**, and **quality** data in a relational structure for analytics.  

**Core Tables:**
- `location` — Province, town, and location type.
- `visits` — Field logs including queue times and assigned employees.
- `water_source` — Water source types and number of people served.
- `employee` — Field workers and surveyors.
- `auditor_report` — Validation from external audits.
- `water_quality` — Surveyor quality scores.
- `well_pollution` — Chemical and biological contamination results.

**Analytical Views:**
- `combined_analysis_table`
- `incorrect_records`

---

## 🔍 Step 1 — Data Integration (Joining Core Tables)

```sql
SELECT
	loc.province_name, 
    loc.town_name,
    v.visit_count,
    v.location_id,
    ws.type_of_water_source,
    ws.number_of_people_served
FROM location AS loc
INNER JOIN visits AS v ON v.location_id = loc.location_id
INNER JOIN water_source AS ws ON ws.source_id = v.source_id;
```

**Insight:**  
This foundational query merges **geographic**, **visitation**, and **water source** data into a unified structure.  
It forms the analytical base for later contamination, queue time, and population coverage analysis.

---

## 📍 Step 2 — Filtering and Refinement  

```sql
SELECT
    loc.province_name, 
    loc.town_name,
    ws.type_of_water_source,
    ws.number_of_people_served
FROM location AS loc
INNER JOIN visits AS v ON v.location_id = loc.location_id
INNER JOIN water_source AS ws ON ws.source_id = v.source_id
WHERE v.visit_count = 1;
```

**Insight:**  
Filtered to retain only **unique visit records per location**, removing duplication or redundant inspection data.  
This ensures accuracy when calculating coverage rates by community and province.

---

## 🧩 Step 3 — Building Analytical Views  

```sql
CREATE VIEW combined_analysis_table AS
SELECT
    water_source.type_of_water_source AS source_type,
    location.town_name,
    location.province_name,
    location.location_type,
    water_source.number_of_people_served AS people_served,
    visits.time_in_queue,
    well_pollution.results
FROM visits
LEFT JOIN well_pollution ON well_pollution.source_id = visits.source_id
INNER JOIN location ON location.location_id = visits.location_id
INNER JOIN water_source ON water_source.source_id = visits.source_id
WHERE visits.visit_count = 1;
```

**Insight:**  
A **view** called `combined_analysis_table` was created for reusable and efficient analysis.  
It consolidates population, queue, and contamination data, simplifying queries for visualization and reporting.

---

## 🌍 Step 4 — Provincial and Town-Level Water Access  

### a️⃣ Provincial Coverage  

```sql
WITH province_totals AS (
  SELECT province_name, SUM(people_served) AS total_ppl_serv
  FROM combined_analysis_table
  GROUP BY province_name
)
SELECT
  ct.province_name,
  ROUND((SUM(CASE WHEN source_type = 'river' THEN people_served ELSE 0 END) * 100.0 / pt.total_ppl_serv), 0) AS river,
  ROUND((SUM(CASE WHEN source_type = 'shared_tap' THEN people_served ELSE 0 END) * 100.0 / pt.total_ppl_serv), 0) AS shared_tap,
  ROUND((SUM(CASE WHEN source_type = 'tap_in_home' THEN people_served ELSE 0 END) * 100.0 / pt.total_ppl_serv), 0) AS tap_in_home
FROM combined_analysis_table ct
JOIN province_totals pt ON ct.province_name = pt.province_name
GROUP BY ct.province_name
ORDER BY ct.province_name;
```

**Insight:**  
This query calculates **percentage access per water source type** by province.  
Findings revealed that **urban provinces** had higher `tap_in_home` access, while **rural regions** relied heavily on `river` and `shared_tap` sources.

---

### b️⃣ Town-Level Access  

```sql
WITH town_totals AS (
  SELECT province_name, town_name, SUM(people_served) AS total_ppl_serv
  FROM combined_analysis_table
  GROUP BY province_name, town_name
)
SELECT
  ct.province_name,
  ct.town_name,
  ROUND((SUM(CASE WHEN source_type = 'river' THEN people_served ELSE 0 END) * 100.0 / tt.total_ppl_serv), 0) AS river,
  ROUND((SUM(CASE WHEN source_type = 'shared_tap' THEN people_served ELSE 0 END) * 100.0 / tt.total_ppl_serv), 0) AS shared_tap,
  ROUND((SUM(CASE WHEN source_type = 'well' THEN people_served ELSE 0 END) * 100.0 / tt.total_ppl_serv), 0) AS well
FROM combined_analysis_table ct
JOIN town_totals tt ON ct.province_name = tt.province_name AND ct.town_name = tt.town_name
GROUP BY ct.province_name, ct.town_name
ORDER BY ct.town_name;
```

**Insight:**  
By disaggregating to town level, disparities emerged — for example, *Mutare* and *Pumula* towns showed **over 60% reliance on wells**, indicating infrastructure expansion priorities.

---

## 🧮 Step 5 — Data Validation (Auditor vs Surveyor Comparison)

```sql
SELECT 
    e.employee_name,
    COUNT(*) AS mistake_count
FROM auditor_report a
JOIN visits v ON a.location_id = v.location_id
JOIN water_quality wq ON v.record_id = wq.record_id
JOIN employee e ON v.assigned_employee_id = e.assigned_employee_id
WHERE a.true_water_source_score != wq.subjective_quality_score
GROUP BY e.employee_name
ORDER BY mistake_count DESC;
```

**Insight:**  
This validation identified mismatched **water source quality scores** between auditors and surveyors.  
Around **8%** of all entries were inconsistent — mostly from a few staff members.  
This enabled **targeted training** and quality control interventions.

---

## 🔎 Step 6 — Fraud & Text Anomaly Detection  

```sql
SELECT *
FROM incorrect_records
WHERE LOWER(remarks) LIKE '%cash%'
   OR LOWER(remarks) LIKE '%gift%';
```

**Insight:**  
Detected potential irregularities in auditor notes or field remarks containing suspicious keywords (“cash”, “gift”).  
This transparency check promoted **ethical reporting** and data accountability.

---

## 💧 Step 7 — Generating Improvement Recommendations  

```sql
SELECT
    l.address,
    l.town_name,
    ws.type_of_water_source,
    wp.results,
    CASE
        WHEN ws.type_of_water_source = 'well' AND wp.results = 'Contaminated: Biological'
          THEN 'Install UV filter'
        WHEN ws.type_of_water_source = 'river'
          THEN 'Drill Well'
        WHEN ws.type_of_water_source = 'shared_tap' AND v.time_in_queue >= 30
          THEN CONCAT('Install ', FLOOR(v.time_in_queue / 30), ' taps nearby')
        WHEN ws.type_of_water_source = 'tap_in_home_broken'
          THEN 'Diagnose local infrastructure'
        ELSE NULL
    END AS improvement
FROM water_source AS ws
LEFT JOIN well_pollution AS wp ON ws.source_id = wp.source_id
INNER JOIN visits AS v ON ws.source_id = v.source_id
INNER JOIN location AS l ON l.location_id = v.location_id
WHERE v.visit_count = 1;
```

**Insight:**  
The query produces **custom improvement suggestions** based on water type and condition:  
- **Contaminated wells** → UV/RO filters  
- **Rivers** → Drilling recommended  
- **Long queues** → More taps needed  
- **Broken taps** → Infrastructure diagnostics  

---

## 🧾 Step 8 — Automating Project Progress Tracking  

```sql
INSERT INTO project_progress (
    source_id, Address, Town, Province, Source_type, Improvement, Source_status
)
SELECT
    ws.source_id,
    l.address,
    l.town_name,
    l.province_name,
    ws.type_of_water_source,
    CASE
        WHEN ws.type_of_water_source = 'well' AND wp.results = 'Contaminated: Chemical'
          THEN 'Install RO filter'
        WHEN ws.type_of_water_source = 'river'
          THEN 'Drill Well'
        WHEN ws.type_of_water_source = 'shared_tap' AND v.time_in_queue >= 30
          THEN CONCAT('Install ', FLOOR(v.time_in_queue / 30), ' taps nearby')
        WHEN ws.type_of_water_source = 'tap_in_home_broken'
          THEN 'Diagnose local infrastructure'
        ELSE NULL
    END AS Improvement,
    'Backlog' AS Source_status
FROM water_source ws
LEFT JOIN well_pollution wp ON ws.source_id = wp.source_id
INNER JOIN visits v ON ws.source_id = v.source_id
INNER JOIN location l ON l.location_id = v.location_id
WHERE v.visit_count = 1;
```

**Insight:**  
Populates a **project tracking table** to monitor implementation status of all recommended interventions.  
This was later visualized in Power BI to track completion rates and timelines.

---

## 📊 Key Findings & Outcomes  

| Focus Area | Insight | Impact |
|-------------|----------|--------|
| **Data Integrity** | 8% mismatch between auditors & surveyors | Improved quality control |
| **Workforce Performance** | 4 employees caused most data errors | Targeted staff retraining |
| **Water Source Reliability** | Wells had the highest contamination rates | Prioritized UV/RO filtering |
| **Infrastructure Gaps** | 40% of rural users depend on rivers/shared taps | Informed drilling & expansion |
| **Operational Delays** | Shared tap queues exceeded 30 mins | Recommended more taps |

---

## 💼 Recruiter Takeaway  

This project showcases **end-to-end SQL analytics and data storytelling**, integrating technical skill with social impact.  

**Highlights:**
- Advanced joins, subqueries, and CTEs.  
- Practical problem-solving for real-world development.  
- Automated reporting with Power BI handoff.  
- Strong data ethics and validation workflows.  

> “From SQL queries to social impact — this analysis demonstrates how data integrity and visualization can improve lives through better water access.”
